### Import dependencies 

In [2]:
pip install opencv-python pandas numpy tqdm


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [59]:
import os

os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
os.environ["OPENCV_FFMPEG_LOGLEVEL"] = "-8"

import sys
import random
import re
import cv2
import numpy as np
import pandas as pd
import contextlib

from pathlib import Path
from datetime import datetime
from collections import deque
from PIL import Image
import imagehash
from tqdm import tqdm

try:
    cv2.setLogLevel(0)
except Exception:
    pass

### Establish paths + settings 

In [77]:
ROOT = Path(r"/Users/alopias/Desktop/deePi-video-processing/converted-videos")
OUT = Path(r"/Users/alopias/Desktop/Huyen-deePi/deePi-laptop")

ACTIVE_FRAME_DIR = OUT / "active_review_batch_004" / "unreviewed"

OUT.mkdir(parents=True, exist_ok=True)
ACTIVE_FRAME_DIR.mkdir(parents=True, exist_ok=True)

In [78]:
# -------------------------
# Settings
# -------------------------
TARGET_ACTIVE_FRAMES = 1500

FPS = 26
MIN_SECONDS_APART = 10
MAX_FRAMES_PER_VID = 6

# Around each random sampled time point, search nearby frames.
# This helps if the exact sampled frame is empty but a nearby frame is active.
SEARCH_WINDOW_SECONDS = 3

# Active-frame thresholds
ACTIVE_EXTRACT_MODE = "strict"  # "broad" or "strict"

if ACTIVE_EXTRACT_MODE == "broad":
    BRIGHTNESS_THRESHOLD = 15
    MIN_BRIGHT_PIXELS = 100
    HASH_THRESHOLD = 5

elif ACTIVE_EXTRACT_MODE == "strict":
    BRIGHTNESS_THRESHOLD = 20
    MIN_BRIGHT_PIXELS = 500
    HASH_THRESHOLD = 8

RANDOM_SEED = 12
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

VIDEO_RE = re.compile(r"^\d{12}\.mp4$")

In [79]:
@contextlib.contextmanager
def suppress_stderr():
    """
    Suppress low-level OpenCV/FFmpeg warnings that print directly to stderr.
    """
    try:
        stderr_fd = sys.stderr.fileno()
    except Exception:
        yield
        return

    with open(os.devnull, "w") as devnull:
        old_stderr_fd = os.dup(stderr_fd)
        try:
            os.dup2(devnull.fileno(), stderr_fd)
            yield
        finally:
            os.dup2(old_stderr_fd, stderr_fd)
            os.close(old_stderr_fd)

### Random sample video 
Use same random sample technique as black frame filter (random sample across 3 cameras, varying time of day, and month).  
Target 500 active frames first --> ~160-170 images per camera 

In [80]:
def parse_video_datetime(video_path):
    """
    Parses filename like 202306222131.mp4
    Into: year=2023, month=6, day=22, hour=21, minute=31
    """
    dt = datetime.strptime(video_path.stem, "%Y%m%d%H%M")

    return {
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
        "hour": dt.hour,
        "minute": dt.minute,
        "datetime": dt,
    }


def build_video_table(root):
    """
    Builds an internal table of videos.
    This is only used inside Python.
    It is not saved as an output CSV.
    """
    rows = []

    for camera_folder in sorted(root.iterdir()):
        if not camera_folder.is_dir():
            continue

        camera = camera_folder.name

        for video_path in sorted(camera_folder.glob("*.mp4")):
            if not VIDEO_RE.match(video_path.name):
                continue

            dt_info = parse_video_datetime(video_path)

            rows.append({
                "camera": camera,
                "video": video_path.name,
                "video_path": video_path,
                **dt_info,
            })

    return pd.DataFrame(rows)


def sample_frame_indices(total_frames, fps):
    """
    Randomly samples candidate frame indices from a video.
    The selected candidate frames are at least MIN_SECONDS_APART apart.
    """
    if total_frames <= 0:
        return []

    min_gap = int(round(fps * MIN_SECONDS_APART))

    if min_gap <= 0:
        min_gap = int(round(FPS * MIN_SECONDS_APART))

    max_offset = min(min_gap - 1, max(total_frames - 1, 0))
    offset = random.randint(0, max_offset)

    possible_frames = list(range(offset, total_frames, min_gap))

    if len(possible_frames) == 0:
        return []

    n_to_sample = min(MAX_FRAMES_PER_VID, len(possible_frames))
    sampled = random.sample(possible_frames, k=n_to_sample)

    return sorted(sampled)


def make_balanced_video_order(camera_df):
    """
    Creates a randomized video order that spreads videos across different hours.
    """
    groups = {}

    for hour, sub_df in camera_df.groupby("hour"):
        records = sub_df.to_dict("records")
        random.shuffle(records)
        groups[hour] = records

    ordered = []

    while groups:
        hours = list(groups.keys())
        random.shuffle(hours)

        for hour in hours:
            if hour not in groups:
                continue

            if len(groups[hour]) == 0:
                del groups[hour]
                continue

            ordered.append(groups[hour].pop())

            if len(groups[hour]) == 0:
                del groups[hour]

    return ordered


def make_camera_targets(camera_names, total_target):
    """
    Splits the total target approximately evenly across cameras.
    """
    camera_names = sorted(list(camera_names))

    base = total_target // len(camera_names)
    remainder = total_target % len(camera_names)

    targets = {}

    for i, camera in enumerate(camera_names):
        targets[camera] = base + (1 if i < remainder else 0)

    return targets

### Active frame filtering 

In [81]:
def is_active_frame(
    frame,
    frame_buffer,
    brightness_threshold=20,
    min_bright_pixels=500,
):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    frame_buffer.append(gray)

    accumulated = gray.copy()

    for old_frame in frame_buffer:
        accumulated = cv2.max(accumulated, old_frame)

    _, thresh = cv2.threshold(
        accumulated,
        brightness_threshold,
        255,
        cv2.THRESH_BINARY
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    bright_pixels = cv2.countNonZero(thresh)

    return bright_pixels > min_bright_pixels

### Get frames

In [82]:
def extract_active_frames_from_video_sample(
    video_row,
    output_dir,
    remaining_target_for_camera,
    fps_fallback=26,
    search_window_seconds=3,
    brightness_threshold=20,
    min_bright_pixels=500,
    hash_threshold=8,
):
    video_path = Path(video_row["video_path"])
    camera = video_row["camera"]
    video_name = video_row["video"]

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Could not open: {video_path}")
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps is None or fps <= 0:
        fps = fps_fallback

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    candidate_frames = sample_frame_indices(total_frames, fps)

    saved_count = 0
    last_hash = None

    half_window = int(round((search_window_seconds * fps) / 2))

    for candidate_frame in candidate_frames:
        if saved_count >= remaining_target_for_camera:
            break

        start_frame = max(0, candidate_frame - half_window)
        end_frame = min(total_frames - 1, candidate_frame + half_window)

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frame_buffer = deque(maxlen=3)
        frame_index = start_frame

        while frame_index <= end_frame:
            ret, frame = cap.read()

            if not ret:
                break

            active = is_active_frame(
                frame,
                frame_buffer=frame_buffer,
                brightness_threshold=brightness_threshold,
                min_bright_pixels=min_bright_pixels,
            )

            if not active:
                frame_index += 1
                continue

            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            current_hash = imagehash.phash(pil_img, hash_size=8)

            if last_hash is not None and (current_hash - last_hash) <= hash_threshold:
                frame_index += 1
                continue

            filename = f"{camera}_{video_name}_frame_{frame_index:05d}.jpg"
            output_path = Path(output_dir) / filename

            cv2.imwrite(str(output_path), frame)

            last_hash = current_hash
            saved_count += 1

            # Save only one active frame from this sampled time window
            break

    cap.release()

    return saved_count

### Balanced extraction from each camera

In [83]:
def extract_active_frames_balanced():
    video_df = build_video_table(ROOT)

    camera_names = sorted(video_df["camera"].unique())

    camera_targets = make_camera_targets(
        camera_names,
        TARGET_ACTIVE_FRAMES
    )

    print()
    print("Target active frames per camera:")
    for camera, target in camera_targets.items():
        print(f"  {camera}: {target}")

    saved_count_by_camera = {camera: 0 for camera in camera_names}

    for camera in camera_names:
        camera_df = video_df[video_df["camera"] == camera].copy()
        video_order = make_balanced_video_order(camera_df)

        target_for_camera = camera_targets[camera]

        progress = tqdm(
            total=target_for_camera,
            desc=f"Saving active frames for {camera}",
            unit="frame",
            file=sys.stdout,
            dynamic_ncols=True,
            leave=True,
        )

        videos_checked = 0

        for video_row in video_order:
            if saved_count_by_camera[camera] >= target_for_camera:
                break

            videos_checked += 1
            remaining = target_for_camera - saved_count_by_camera[camera]

            with suppress_stderr():
                saved = extract_active_frames_from_video_sample(
                    video_row=video_row,
                    output_dir=ACTIVE_FRAME_DIR,
                    remaining_target_for_camera=remaining,
                    fps_fallback=FPS,
                    search_window_seconds=SEARCH_WINDOW_SECONDS,
                    brightness_threshold=BRIGHTNESS_THRESHOLD,
                    min_bright_pixels=MIN_BRIGHT_PIXELS,
                    hash_threshold=HASH_THRESHOLD,
                )

            saved_count_by_camera[camera] += saved

            if saved > 0:
                progress.update(saved)

            progress.set_postfix({
                "saved": saved_count_by_camera[camera],
                "target": target_for_camera,
                "videos_checked": videos_checked,
                "last_saved": saved,
            })
            progress.refresh()

        print(
            f"Finished {camera}: "
            f"{saved_count_by_camera[camera]} / {target_for_camera}"
        )

    print()
    print("Final active-frame counts:")
    for camera, count in saved_count_by_camera.items():
        print(f"  {camera}: {count} / {camera_targets[camera]}")

    print()
    print(f"Saved frames folder: {ACTIVE_FRAME_DIR}")

    return saved_count_by_camera

### Run

In [84]:
camera_counts = extract_active_frames_balanced()

Saving active frames for 10.0.11.2:  66%|██████▌   | 331/500 [1:33:04<35:16, 12.52s/frame, saved=331, target=500, videos_checked=1132, last_saved=0]  

[h264 @ 0x13ce99660] cabac_init_idc 32 overflow
[h264 @ 0x13ce99660] decode_slice_header error
[h264 @ 0x13ce99660] no frame!


Saving active frames for 10.0.11.2:  99%|█████████▉| 495/500 [2:20:25<00:43,  8.73s/frame, saved=495, target=500, videos_checked=1707, last_saved=0]  

[h264 @ 0x13cd555f0] error while decoding MB 1 60, bytestream -10


Saving active frames for 10.0.11.2: 100%|██████████| 500/500 [2:22:14<00:00, 17.07s/frame, saved=500, target=500, videos_checked=1727, last_saved=1]


[h264 @ 0x13ce99c40] error while decoding MB 54 46, bytestream -6


[h264 @ 0x13cd78b70] error while decoding MB 47 57, bytestream -5


Finished 10.0.12.2: 431 / 500
Saving active frames for 10.0.16.2:  19%|█▉        | 97/500 [11:50<59:55,  8.92s/frame, saved=97, target=500, videos_checked=180, last_saved=0]  

[h264 @ 0x13ce926b0] error while decoding MB 117 66, bytestream -6


Saving active frames for 10.0.16.2: 100%|██████████| 500/500 [1:01:08<00:00, 13.16s/frame, saved=500, target=500, videos_checked=943, last_saved=1]Finished 10.0.16.2: 500 / 500

Final active-frame counts:
  10.0.11.2: 500 / 500
  10.0.12.2: 431 / 500
  10.0.16.2: 500 / 500

Saved frames folder: /Users/alopias/Desktop/Huyen-deePi/deePi-laptop/active_review_batch_004/unreviewed
Saving active frames for 10.0.16.2: 100%|██████████| 500/500 [1:01:08<00:00,  7.34s/frame, saved=500, target=500, videos_checked=943, last_saved=1]
